# SemantyFish Cleanup

## Imports, file loading, inspection

In [528]:
import json, pprint, re
import pandas as pd

In [529]:
with open("all_species_data.json", encoding="utf-8") as f:
    df = json.load(f)

pprint.pprint(df[0])

{'air_breathing_status': 'WaterAssumed',
 'aquarium_demand': {'details': None, 'value': 'never/rarely'},
 'body_shape': 'fusiform / normal',
 'brackish_water_environment': True,
 'catching_methods': {'main_method': 'seines',
                      'main_method_using_fao_name': None,
                      'other_methods': ['gillnets',
                                        'other method',
                                        'traps',
                                        'seines',
                                        'trawls']},
 'comments': 'Adults occur in a wide variety of freshwater habitats like '
             'rivers, lakes, sewage canals and irrigation channels (Ref. '
             '28714). They do not do well in pure salt water, but  able to '
             'survive in brackish water (Ref. 52307). Mainly diurnal. They '
             'feed mainly on phytoplankton or benthic algae. Additionally, '
             'insect larvae are of some importance, as are aufwuchs and '
   

In [530]:
for dim in df[0]['dimensions']:
    print(dim)

{'type': 'most shallow waters', 'value': 0.0, 'unit': 'meters', 'measurement_type': None, 'sex': None}
{'type': 'max length', 'value': 60.0, 'unit': 'centimeters', 'measurement_type': 'Standard Length', 'sex': 'male or unsexed'}
{'type': 'longevity wild', 'value': 9.0, 'unit': 'years', 'measurement_type': None, 'sex': None}
{'type': 'most deep waters', 'value': 20.0, 'unit': 'meters', 'measurement_type': None, 'sex': None}
{'type': 'max weight', 'value': 4324.0, 'unit': 'kilograms', 'measurement_type': None, 'sex': 'male or unsexed'}
{'type': 'common deep waters', 'value': 20.0, 'unit': 'meters', 'measurement_type': None, 'sex': None}
{'type': 'longevity captive', 'value': 12.0, 'unit': 'years', 'measurement_type': None, 'sex': None}


## Creating new df based on schema


In [531]:
test_df = pd.DataFrame(columns=['pet_id', 'pet_scientific_name','pet_vernacular_name','pet_genus', 'pet_family',
                        'pet_body_shape', 'pet_traits', 'pet_max_length', 'pet_max_weight', 'pet_longevity', 'pet_habitat',
                        'pet_temperature', 'pet_ph_range', 'pet_water_hardness', 'pet_tank_size',
                        'pet_migration_type', 'pet_danger', 'pet_is_native', 'pet_comments', 'pet_aquarium'])

In [532]:
test_df.dtypes

pet_id                 object
pet_scientific_name    object
pet_vernacular_name    object
pet_genus              object
pet_family             object
pet_body_shape         object
pet_traits             object
pet_max_length         object
pet_max_weight         object
pet_longevity          object
pet_habitat            object
pet_temperature        object
pet_ph_range           object
pet_water_hardness     object
pet_tank_size          object
pet_migration_type     object
pet_danger             object
pet_is_native          object
pet_comments           object
pet_aquarium           object
dtype: object

## Adding pet ids

In [533]:
# Add pet_ids to test_df based on 'id' field in df
test_df['pet_id'] = [item['id'] for item in df]

# Id might also be used across other db, appending 'f' to indicate fishbase id
test_df['pet_id'] = test_df['pet_id'].astype(str) + 'f'

## Adding names

In [534]:
# Adding scientific name
test_df['pet_scientific_name'] = [item['scientific_name'] for item in df]

# Adding vernacular name, capitalising first letter of each word
# Leave missing common names as NaN
test_df['pet_vernacular_name'] = [item.get('common_name', None).title() if item.get('common_name') else None for item in df]  

## Adding family/genus

In [535]:
# Add genus and family based on 'genus' and 'family' fields in df, getting just genus_name/family_name from the inner dict
test_df['pet_genus'] = [item['genus']['genus_name'].title() for item in df]
test_df['pet_family'] = [item['family']['family_name'].title() for item in df]

## Adding body shape

In [536]:
# Add body shape
test_df['pet_body_shape'] = [item['body_shape'].title() for item in df]

## Adding measurements (length, weight, longevity)

In [537]:
# from the 'dimensions' field, extract the unique units for the inner dicts in the list where the 'type' is 'max length', 'max weight', 'longevity wild', or 'longevity captive'
units = set()
for item in df:     
    for dim in item['dimensions']:
        if dim['type'] in ['max length', 'max weight', 'longevity wild', 'longevity captive']:
            units.add(dim['unit'])
print(units)

{'kilograms', 'years', 'centimeters'}


In [538]:
# Add max length (cm), weight (kg), longevity (years)
# for max length/weight, retrieve the inner dict from 'dimensions' where 'type' = 'max_length', then get 'value' from that dict
test_df['pet_max_length'] = [next((dim['value'] for dim in item['dimensions'] if dim['type'] == 'max length'), None) for item in df]
test_df['pet_max_weight'] = [next((dim['value']/1000 for dim in item ['dimensions'] if dim['type'] == 'max weight'), None) for item in df]

# for longevity, retrieve the inner dict from 'dimensions' where 'type' = 'longevity_wild' or 'longevity_captive', 
# preferring the greater of the two values
def get_longevity(dimensions):
    longevity_wild = next((dim['value'] for dim in dimensions if dim['type'] == 'longevity wild'), None)
    longevity_captive = next((dim['value'] for dim in dimensions if dim['type'] == 'longevity captive'), None)
    if longevity_wild is not None and longevity_captive is not None:
        return max(longevity_wild, longevity_captive)
    elif longevity_captive is not None:
        return longevity_captive
    elif longevity_wild is not None:
        return longevity_wild
    else:
        return None
test_df['pet_longevity'] = [get_longevity(item['dimensions']) for item in df]



## Adding migration type

In [539]:
# Add migration type
test_df['pet_migration_type'] = [item['migration_type'].title() if item['migration_type'] else None for item in df]

## Adding pet danger

In [540]:
danger = set()
for item in df:
    danger.add(item['dangerous_species_indicator'])
print(danger)

{'poisonous to eat', 'reports of ciguatera poisoning', None, 'potential pest', 'traumatogenic', 'venomous', 'other', 'harmless'}


In [541]:
electric = set()
for item in df:
    electric.add(item['electrogenic'])
print(electric)

{'no special ability', 'electrosensing only', None, 'Weakly discharging', 'Electrosensing only', 'weakly discharging', 'strongly discharging'}


In [542]:
# pet_danger -->
# harmless: harmless, potential pest
# poisonous: poisonous to eat, reports of ciguatera poisoning
# venomous: venomous
# aggressive: traumatogenic
# unknown: null, None, other
def classify_danger(item):
    dangerStr = ''
    # Evaluate dangerous_species_indicator first
    if item['dangerous_species_indicator'] is None:
        dangerStr = 'Unknown'
    elif item['dangerous_species_indicator'].lower() in ['harmless', 'potential pest']:
        dangerStr = 'Harmless'
    elif item['dangerous_species_indicator'].lower() in ['poisonous to eat', 'reports of ciguatera poisoning']:
        dangerStr = 'Poisonous'
    elif item['dangerous_species_indicator'].lower() == 'venomous':
        dangerStr = 'Venomous'
    elif item['dangerous_species_indicator'].lower() == 'traumatogenic':
        dangerStr = 'Aggressive'


    # Evaluate electrogenic
    if item['electrogenic'] is None:
        dangerStr += '; Unknown Electrogenic'
    else:
        dangerStr += f"; {item['electrogenic'].title()}"

    return dangerStr

test_df['pet_danger'] = [classify_danger(item) for item in df]

## Extracting temperature from comments


In [543]:
import re
def get_temp(comment):
    if comment is None:
        return None
    
    matches = re.findall(r'(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*°([CFcf])', comment)

    if not matches:
        return None

    ranges = []
    for low, high, unit in matches:
        
        width = float(high) - float(low)
        ranges.append((low, high, unit.upper(), width))

    # Pick range with smallest lower bound
    smallest = min(ranges, key=lambda x: x[3])  # x[3] = width of the range

    #print(f"{smallest[0]}-{smallest[1]} °{smallest[2]}")
    return(f"{smallest[0]}-{smallest[1]} °{smallest[2]}")

test_df['pet_temperature'] = [get_temp(item['comments']) for item in df]

## Adding comments

In [544]:
# Add comments, remove any "Ref. some number" from the original comment
def clean_comments(comment):
    if comment is None:
        return None
    # Remove "Ref. some number" using regex
    cleaned_comment = re.sub(r'\(Ref\.\s*\d+(?:,\s*\d+)*\)', '', comment)
    return cleaned_comment.strip()

test_df['pet_comments'] = [clean_comments(item['comments']) for item in df]

# Adding missing fish

In [545]:
with open("failed_species_names.json", encoding="utf-8") as f:
    missing_json = json.load(f)

pprint.pprint(missing_json[0])

{'id': 1554, 'name': 'Laeviscutella dekimpei'}


In [546]:
# Convert to df
missing_df = pd.DataFrame(missing_json)
missing_df = missing_df.rename(columns={'id': 'pet_id', 'name': 'pet_scientific_name'})
missing_df['pet_id'] = missing_df['pet_id'].astype(str) + 'f'

# Check if any of the missing scientific names have trailing text, i.e. "Bairdiella goeldi : fisheries" and remove it if so
def clean_scientific_name(name):
    if name is None:
        return None
    if ':' in name:
        return name.split(':')[0].strip()
    return name.strip()

missing_df['pet_scientific_name'] = missing_df['pet_scientific_name'].apply(clean_scientific_name)

# Add contents of missing_df to test_df, adding 'f' to the end to indicate fishbase id
test_df = pd.concat([test_df, missing_df], ignore_index=True)

In [547]:
# Setting native/non-native
test_df['pet_is_native'] = 'Not Native'

# Compare fish.csv against Species in fish.csv; if the fish is found in fish.csv AND the Occurrence == 'native', set pet_is_native to 'Native'
native_fish_df = pd.read_csv('fish.csv')
native_fish_df = native_fish_df[native_fish_df['Occurrence'].isin(['native', 'endemic'])]

test_df['pet_is_native'] = test_df.apply(lambda row: 'Native' if ((native_fish_df['Species'].str.lower() == row['pet_scientific_name'].lower()).any()) else 'Not Native', axis=1)

In [548]:
# Output to csv the fish that are in native_fish_df but not in test_df; these are probably saltwater fish so disregard
native_fish_not_in_test_df = native_fish_df[~native_fish_df['Species'].isin(test_df['pet_scientific_name'])]
native_fish_not_in_test_df.to_csv('native_fish_not_in_test_df.csv', index=False)

# Cleaning common aquatic df

In [549]:
# Adding info from common_aquatic.csv - indicator if this species is commonly kept as an aquatic pet
common_aquatic_df = pd.read_csv('common_aquatic.csv', encoding="mbcs")

print(common_aquatic_df.columns)
common_aquatic_df.head(10)

Index(['Common name', 'Scientific name', 'Image', 'Size', 'Remarks',
       'Tank size', 'Temperature range', 'pH range', 'Water hardness'],
      dtype='object')


,Common name,Scientific name,Image,Size,Remarks,Tank size,Temperature range,pH range,Water hardness
0,Brown-Point Shield Skin,Aspidoras fuscoguttatus,NaN,3.8 cm\n(1.5 in),NaN,30 Gallons,22–25 °C (72–77 °F)[1],5.5–6.8[1],NaN
1,Aspidoras Cory-Cat; Pygmy Aspidoras,Aspidoras lakoi,NaN,4 cm (1.6 in)[2],NaN,28 Gallons,22–25 °C (72–77 °F),6.5-7.5,NaN
2,Sixray corydoras; false corydoras,Aspidoras pauciradiatus,NaN,2.9 cm (1.1 in)[3],NaN,20 Gallons,23–28 °C,6.0-7.2[3],NaN
3,Loach catfish,Aspidoras rochai,NaN,4 cm (1.6 in) maximum length[4],NaN,20 Gallons,21–25°C[4],6.0-7.5[4],NaN
4,Britski's catfish,Corydoras britskii,NaN,8.9 cm (3.5 in),NaN,NaN,20–24 °C (68–75 °F),6.5-7.2[5],NaN
5,Hognosed brochis,Corydoras multiradiatus,NaN,6.6 cm (2.6 in),NaN,NaN,21–24 °C,6.0–7.2,NaN
6,Emerald catfish,Corydoras splendens,NaN,10 cm (3.9 in),The Emerald Cory Catfish is a very hardy and r...,20 Gallons,22 – 27.7 °C,5.8-8.0[7],NaN
7,Cascarudo,Callichthys callichthys,NaN,20 cm (7.9 in),NaN,NaN,18–28 °C,5.8–8.3,NaN
8,Blacktop corydoras,Corydoras acutus,NaN,4.4 cm (1.7 in),NaN,NaN,22–28 °C[8],6.0–7.5[8],NaN
9,Adolfo's catfish/corydoras,Corydoras adolfoi,NaN,5.7 cm (2.2 in),NaN,NaN,22–26 °C,6.0–7.0,NaN


In [550]:
# Match case of vernacular names
common_aquatic_df['Common name'] = common_aquatic_df['Common name'].str.title()

In [551]:
# In Size column, remove everything after first instance of 'cm' including the 'cm', and strip any whitespace
def clean_size(size):
    if pd.isna(size):
        return None
    if 'cm' in size:
        return float(size.split('cm')[0].strip())
    return float(size.strip())
common_aquatic_df['Size'] = common_aquatic_df['Size'].apply(clean_size)

In [552]:
# In Remarks columns, remove any text in square brackets including the brackets, and strip any whitespace
def clean_remarks(remarks):
    if pd.isna(remarks):
        return None
    cleaned_remarks = re.sub(r'\[.*?\]', '', remarks).strip()
    return cleaned_remarks
common_aquatic_df['Remarks'] = common_aquatic_df['Remarks'].apply(clean_remarks)

In [553]:
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.max_colwidth', 200,
                       'display.width', 1000):
    print(common_aquatic_df[common_aquatic_df['Scientific name'] == 'Corydoras splendens']['Remarks'])

6    The Emerald Cory Catfish is a very hardy and resilient fish.  Disease should not be a concern provided that you maintain the aquarium to standards.
Name: Remarks, dtype: object


In [554]:
# In Tank size column, leave only the first encountered number
def clean_tank_size(tank_size):
    if pd.isna(tank_size):
        return None
    match = re.search(r'(\d+)', tank_size)
    if match:
        return int(match.group(1))
    return None
common_aquatic_df['Tank size'] = common_aquatic_df['Tank size'].apply(clean_tank_size)

In [555]:
# In Temperature range column, remove everything after "°C", and reformat the numbers to be in the form "low-high °C"
def clean_temperature_range(temp_range):
    if pd.isna(temp_range):
        return None
    match = re.search(r'(\d+(?:\.\d+)?)\s*[–-]\s*(\d+(?:\.\d+)?)\s*°C', temp_range)
    if match:
        low, high = match.groups()
        return f"{low}-{high} °C"
    return temp_range.strip()
common_aquatic_df['Temperature range'] = common_aquatic_df['Temperature range'].apply(clean_temperature_range)

In [556]:
# In pH range column, remove anything after and including the first instance of a square bracket, and get the range in the form "low-high"
def clean_ph_range(ph_range):
    if pd.isna(ph_range):
        return None
    cleaned_ph_range = re.sub(r'\[.*?\]', '', ph_range).strip()
    match = re.search(r'(\d+(?:\.\d+)?)\s*[–-]\s*(\d+(?:\.\d+)?)', cleaned_ph_range)
    if match:
        low, high = match.groups()
        return f"{low}-{high}"
    return cleaned_ph_range.strip()
common_aquatic_df['pH range'] = common_aquatic_df['pH range'].apply(clean_ph_range)

In [557]:
# In Water hardness column, get the range in the form "low-high"
def clean_water_hardness(hardness):
    if pd.isna(hardness):
        return None
    match = re.search(r'(\d+(?:\.\d+)?)\s*[–-]\s*(\d+(?:\.\d+)?)', hardness)
    if match:
        low, high = match.groups()
        return f"{low}-{high}"
    return hardness.strip()
common_aquatic_df['Water hardness'] = common_aquatic_df['Water hardness'].apply(clean_water_hardness)

In [558]:
common_aquatic_df.head(20)

,Common name,Scientific name,Image,Size,Remarks,Tank size,Temperature range,pH range,Water hardness
0,Brown-Point Shield Skin,Aspidoras fuscoguttatus,NaN,3.80,None,30.0,22-25 °C,5.5-6.8,None
1,Aspidoras Cory-Cat; Pygmy Aspidoras,Aspidoras lakoi,NaN,4.00,None,28.0,22-25 °C,6.5-7.5,None
2,Sixray Corydoras; False Corydoras,Aspidoras pauciradiatus,NaN,2.90,None,20.0,23-28 °C,6.0-7.2,None
3,Loach Catfish,Aspidoras rochai,NaN,4.00,None,20.0,21-25 °C,6.0-7.5,None
4,Britski'S Catfish,Corydoras britskii,NaN,8.90,None,NaN,20-24 °C,6.5-7.2,None
5,Hognosed Brochis,Corydoras multiradiatus,NaN,6.60,None,NaN,21-24 °C,6.0-7.2,None
6,Emerald Catfish,Corydoras splendens,NaN,10.00,The Emerald Cory Catfish is a very hardy and r...,20.0,22-27.7 °C,5.8-8.0,None
7,Cascarudo,Callichthys callichthys,NaN,20.00,None,NaN,18-28 °C,5.8-8.3,None
8,Blacktop Corydoras,Corydoras acutus,NaN,4.40,None,NaN,22-28 °C,6.0-7.5,None
9,Adolfo'S Catfish/Corydoras,Corydoras adolfoi,NaN,5.70,None,NaN,22-26 °C,6.0-7.0,None


In [559]:
test_df[test_df['pet_id'] == '70190f']

,pet_id,pet_scientific_name,pet_vernacular_name,pet_genus,pet_family,pet_body_shape,pet_traits,pet_max_length,pet_max_weight,pet_longevity,pet_habitat,pet_temperature,pet_ph_range,pet_water_hardness,pet_tank_size,pet_migration_type,pet_danger,pet_is_native,pet_comments,pet_aquarium
20014,70190f,Bairdiella goeldi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not Native,NaN,NaN


## Adding common aquatic additional information

In [560]:
test_df.columns

Index(['pet_id', 'pet_scientific_name', 'pet_vernacular_name', 'pet_genus',
       'pet_family', 'pet_body_shape', 'pet_traits', 'pet_max_length',
       'pet_max_weight', 'pet_longevity', 'pet_habitat', 'pet_temperature',
       'pet_ph_range', 'pet_water_hardness', 'pet_tank_size',
       'pet_migration_type', 'pet_danger', 'pet_is_native', 'pet_comments',
       'pet_aquarium'],
      dtype='object')

In [561]:
common_aquatic_df.columns

Index(['Common name', 'Scientific name', 'Image', 'Size', 'Remarks',
       'Tank size', 'Temperature range', 'pH range', 'Water hardness'],
      dtype='object')

In [567]:
# Matching on test_df["pet_scientific_name"] == common_aquatic_df["Scientific name"], do the following:
# - For the matching, ignore case and whitespace differences in the scientific name
# - If there's a match, set test_df['pet_aquarium'] to True, else False
# - Update test_df["pet_vernacular_name"] to include any common names that don't already exist in test_df["pet_vernacular_name"] from common_aquatic_df["Common name"], seperated by a semicolon
# - For each match, if there is an existing pet_max_length in test_df, pick the larger of the two between test_df["pet_max_length"] and common_aquatic_df['Size']
# - For pet_temperature, if test_df already has a value, keep it; else, use common_aquatic_df['Temperature range']
# - For pet_ph_range, use common_aquatic_df['pH range']
# - For pet_water_hardness, use common_aquatic_df['Water hardness']
# - Update test_df['pet_tank_size'] to be common_aquatic_df['Tank size'] if common_aquatic_df['Tank size'] is not null
# - Append any remarks from common_aquatic_df['Remarks'] to test_df['pet_comments'], seperated by a newline

def update_test_df(row):
    match = common_aquatic_df[common_aquatic_df['Scientific name'].str.lower().str.strip() == row['pet_scientific_name'].lower().strip()]
    if not match.empty:
        row['pet_aquarium'] = True
        
        common_name = match.iloc[0]['Common name']
        if pd.notna(common_name):
            if pd.isna(row['pet_vernacular_name']):
                row['pet_vernacular_name'] = common_name
            elif common_name not in row['pet_vernacular_name']:
                row['pet_vernacular_name'] += f"; {common_name}"
        
        size = match.iloc[0]['Size']
        if pd.notna(size):
            if pd.notna(row['pet_max_length']):
                row['pet_max_length'] = max(row['pet_max_length'], size)
            else:
                row['pet_max_length'] = size
        
        temp_range = match.iloc[0]['Temperature range']
        if pd.notna(temp_range) and pd.isna(row['pet_temperature']):
            row['pet_temperature'] = temp_range
        
        ph_range = match.iloc[0]['pH range']
        if pd.notna(ph_range):
            row['pet_ph_range'] = ph_range
        
        hardness = match.iloc[0]['Water hardness']
        if pd.notna(hardness):
            row['pet_water_hardness'] = hardness
        
        tank_size = match.iloc[0]['Tank size']
        if pd.notna(tank_size):
            row['pet_tank_size'] = tank_size
        
        remarks = match.iloc[0]['Remarks']
        if pd.notna(remarks):
            if pd.isna(row['pet_comments']):
                row['pet_comments'] = remarks
            elif remarks not in row['pet_comments']:
                row['pet_comments'] += f"\n{remarks}"
    else:
        row['pet_aquarium'] = False
    
    return row

test_df = test_df.apply(update_test_df, axis=1)


In [568]:
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.max_colwidth', 200,
                       'display.width', 1000):
    print(test_df[test_df["pet_aquarium"] == True][['pet_scientific_name']].sort_values('pet_scientific_name', ascending=True)) 



                  pet_scientific_name
3789           Abramites hypselonotus
9097                Acanthicus adonis
5551             Acantopsis dialuzona
4743             Acarichthys heckelii
5704         Acestrorhynchus falcatus
6149        Achiroides melanorhynchus
1459               Acipenser ruthenus
5640       Acrochordonichthys rugosus
4619            Agamyxis pectinifrons
7696               Ageneiosus inermis
8018           Aguarunichthys torosus
6011                  Akysis prashadi
3999        Alestopetersius brichardi
4000         Alestopetersius caudalis
7175                 Alfaro cultratus
16611        Allenbatrachus grunniens
2479           Altolamprologus calvus
2480    Altolamprologus compressiceps
11183              Amatitlania myrnae
1355        Amatitlania nigrofasciata
5205               Amatitlania sajica
3864            Ambastaia sidthimunki
763                        Amia calva
1553          Amphilophus citrinellus
90                 Anabas testudineus
2397        

In [572]:
test_df[test_df["pet_scientific_name"] == "Astronotus ocellatus"]

,pet_id,pet_scientific_name,pet_vernacular_name,pet_genus,pet_family,pet_body_shape,pet_traits,pet_max_length,pet_max_weight,pet_longevity,pet_habitat,pet_temperature,pet_ph_range,pet_water_hardness,pet_tank_size,pet_migration_type,pet_danger,pet_is_native,pet_comments,pet_aquarium
1352,3612f,Astronotus ocellatus,Oscar,Astronotus,Cichlidae,Short And / Or Deep,NaN,45.7,1.58,NaN,NaN,22-27 °C,6.0-7.5,NaN,NaN,None,Harmless; No Special Ability,Not Native,Preferably inhabits quiet shallow waters in mu...,True


In [576]:
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.max_colwidth', 1200,
                       'display.width', 1000):
    pprint.pprint(test_df[test_df["pet_scientific_name"] == "Astronotus ocellatus"]["pet_comments"]) 

1352    Preferably inhabits quiet shallow waters in mud-bottomed and sand-bottomed canals and ponds .  Feeds on small fish, crayfish, worms and insect larvae.  Quite popular with aquarists but not for aquaculturists because of its slow growth .  Maximum length 40 cm TL .  A highly esteemed food fish in South America .\nMany people that purchase these fish do not realize that the fish could grow to a foot long (30 cm) within a year. Due to their fast growth rate and large size as an adult, they are often kept in aquariums that are too small for them.
Name: pet_comments, dtype: object


## Setting pet_habitat to aquatic


In [579]:
# Set pet_habitat to aquatic in all rows
test_df['pet_habitat'] = 'Aquatic'

# Final Output

In [581]:
# Output to csv
test_df.to_csv('cleaned_fish_data.csv', index=False)

In [580]:
###TEST CELL
test_df.head(10)

,pet_id,pet_scientific_name,pet_vernacular_name,pet_genus,pet_family,pet_body_shape,pet_traits,pet_max_length,pet_max_weight,pet_longevity,pet_habitat,pet_temperature,pet_ph_range,pet_water_hardness,pet_tank_size,pet_migration_type,pet_danger,pet_is_native,pet_comments,pet_aquarium
0,2f,Oreochromis niloticus,Nile Tilapia,Oreochromis,Cichlidae,Fusiform / Normal,NaN,60.0,4.324,12.0,Aquatic,13.5-33 °C,NaN,NaN,NaN,Potamodromous,Harmless; No Special Ability,Not Native,Adults occur in a wide variety of freshwater h...,False
1,3f,Oreochromis mossambicus,Mozambique Tilapia,Oreochromis,Cichlidae,Fusiform / Normal,NaN,39.0,1.130,11.0,Aquatic,17-35 °C,NaN,NaN,NaN,Amphidromous,Harmless; No Special Ability,Not Native,Adults thrive in standing waters . Inhabits re...,False
2,35f,Anguilla anguilla,European Eel,Anguilla,Anguillidae,Eel-Like,NaN,133.0,6.599,55.0,Aquatic,20-25 °C,NaN,NaN,NaN,Catadromous,Harmless; No Special Ability,Not Native,Inhabits all types of benthic habitats from st...,False
3,63f,Dicentrarchus labrax,European Seabass,Dicentrarchus,Moronidae,Fusiform / Normal,NaN,103.0,12.000,30.0,Aquatic,None,NaN,NaN,NaN,Catadromous,Harmless; No Special Ability,Not Native,"Adults manifest demersal behavior, inhabit coa...",False
4,73f,Gobius paganellus,Rock Goby,Gobius,Gobiidae,Elongated,NaN,13.0,NaN,10.0,Aquatic,None,NaN,NaN,NaN,Amphidromous,Harmless; No Special Ability,Not Native,"Predominantly marine, but may enter freshwater...",False
5,79f,Ctenopharyngodon idella,Grass Carp,Ctenopharyngodon,Xenocyprididae,Fusiform / Normal,NaN,150.0,45.000,21.0,Aquatic,None,NaN,NaN,NaN,Potamodromous,Harmless; No Special Ability,Not Native,"Adults occur in lakes, ponds, pools and backwa...",False
6,80f,Chanos chanos,Milkfish,Chanos,Chanidae,Elongated,NaN,124.0,14.000,15.0,Aquatic,None,NaN,NaN,NaN,Amphidromous,Harmless; No Special Ability,Native,Adults are found in offshore marine waters and...,False
7,82f,Labeo rohita,Roho Labeo,Labeo,Cyprinidae,Fusiform / Normal,NaN,200.0,45.000,10.0,Aquatic,None,NaN,NaN,NaN,Potamodromous,Harmless; No Special Ability,Not Native,Adults inhabit rivers . A diurnal species and...,False
8,101f,Alosa alosa,Allis Shad,Alosa,Alosidae,Fusiform / Normal,NaN,83.0,4.000,10.0,Aquatic,None,NaN,NaN,NaN,Anadromous,Harmless; No Special Ability,Not Native,"Amphihaline species, schooling and strongly mi...",False
9,105f,Alosa immaculata,Pontic Shad,Alosa,Alosidae,Fusiform / Normal,NaN,39.0,NaN,7.0,Aquatic,6-9 °C,NaN,NaN,NaN,Anadromous,Harmless; No Special Ability,Not Native,"Thai species is pelagic at sea, in deep water....",False
